### Case 2-FB: Al–hydrolysis

> **Title:** *Decision-free aqueous speciation with precipitation: Migrating from iterative if-loops to closed-form Tableau using Fischer–Burmeister and piecewise constraints*  
>
> **Authors:**  
> - **Claudia Santiviago**$^{a,d}$  
> - **Wayne Parker**$^b$  
> - **Scott Smith**$^c$  
>
> **Affiliations:**  
> $^a$ Biotechnological Processes for the Environment Group, Faculty of Engineering, Universidad de la República, Uruguay.  
> $^b$ Department of Civil and Environmental Engineering, University of Waterloo, Canada.  
> $^c$ Department of Chemistry and Biochemistry, Wilfrid Laurier University, Canada.  
>
> $^d$ **Corresponding author:** [csantiviago@fing.edu.uy](mailto:csantiviago@fing.edu.uy)

---

#### User instructions

This notebook is configured to run the **pre-defined pH–Al<sub>T</sub> grid** used in the manuscript (error-surface / sensitivity analysis).  
No user changes are required.


In [13]:
% =========================
% Setup for Aluminum Case
% =========================
clc; clear all; warning off; close all;

% 1. Force Gnuplot
graphics_toolkit("gnuplot");

% 2. Configure PNG Driver (HD Resolution)
setenv("GNUTERM", "png size 1200,900 font 'Arial,14' linewidth 2");

% 3. Octave Styles
set(0, "defaultlinelinewidth", 2);
set(0, "defaultaxesfontsize", 14);
set(0, "defaulttextfontsize", 14);
set(0, "defaultaxesfontweight", "bold");
set(0, "defaultaxesgridalpha", 0.5);

% 4. Simulation Inputs
AlT = 1.0e-5;
pHvals = 2 : 0.1 : 12;

fprintf('Setup complete. Ready to solve.\n');

Setup complete. Ready to solve.


In [14]:
% Call the solver
% (Ensure Al_solid_complementarity_FB.m is in the folder)
[results, results2D] = Al_solid_complementarity_FB(AlT, pHvals);

fprintf('Calculation finished.\n');

Calculation finished.


In [15]:
fprintf('Generating Aluminum figures...\n');

% --- Helper to save images ---
function save_al_fig(h, name)
    set(h, 'PaperUnits', 'inches');
    set(h, 'PaperPosition', [0 0 12 9]); % 12x9 inches
    set(h, 'PaperSize', [12 9]);
    print(h, name, "-dpng", "-r120");
    close(h);
end

% === FIG 1: 1D Subplots ===
h1 = figure(1, 'visible', 'off');

% (a) Aqueous
subplot(2,2,1);
iAl3 = find(strcmp(results.species,'Al^{3+}'));
iAlOH4 = find(strcmp(results.species,'Al(OH)4-'));
iAl13 = find(strcmp(results.species,'Al13(OH)32^{7+}'));
if ~isempty(iAl3), plot(results.pH, results.Caq(iAl3,:)./AlT, '-b','DisplayName','Al^{3+}'); hold on; end
if ~isempty(iAlOH4), plot(results.pH, results.Caq(iAlOH4,:)./AlT, '-m','DisplayName','Al(OH)_4^-'); end
if ~isempty(iAl13), plot(results.pH, results.Caq(iAl13,:)./AlT, '-r','DisplayName','Al_{13}'); end
xlabel('pH'); ylabel('Fraction Al'); grid on; legend('Location','best'); title('(a) Main aqueous species');

% (b) Solid
subplot(2,2,2);
plot(results.pH, results.xcp./AlT, '-ok','MarkerSize',3);
xlabel('pH'); ylabel('Frac. Solid'); grid on; title('(b) Al(OH)_3(s)');

% (c) Mass Error
subplot(2,2,3);
semilogy(results.pH, results.massErr, '-r');
xlabel('pH'); ylabel('Error (M)'); grid on; title('(c) Mass Balance Error');

% (d) SI
subplot(2,2,4);
plot(results.pH, results.SI, '-b');
xlabel('pH'); ylabel('SI'); grid on; title('(d) Saturation Index');

save_al_fig(h1, "Fig_Al_1.png");


% === FIG 2: Semilogy Species ===
h2 = figure(2, 'visible', 'off');
iAlOH3plus = find(strcmp(results.species,'Al(OH)^{2+}'));
iAl2 = find(strcmp(results.species,'Al2(OH)2^{4+}'));
iAl3 = find(strcmp(results.species,'Al3(OH)4^{5+}'));

semilogy(results.pH, results.Caq(iAl3,:)./AlT, 'DisplayName', 'Al^{3+}'); hold on;
semilogy(results.pH, results.Caq(iAlOH3plus,:)./AlT, 'DisplayName', 'Al(OH)^{2+}');
semilogy(results.pH, results.Caq(iAlOH4,:)./AlT, 'DisplayName', 'Al(OH)_4^-');
semilogy(results.pH, results.Caq(iAl13,:)./AlT, 'DisplayName', 'Al_{13}', 'LineWidth', 2);
semilogy(results.pH, results.Caq(iAl2,:)./AlT, 'DisplayName', 'Al_2');
semilogy(results.pH, results.xcp./AlT, 'k--', 'DisplayName', 'Solid');

legend('Location','northeastoutside');
xlabel('pH'); ylabel('Concentration (M)'); title('Species Distribution (Log Scale)');
ylim([1e-10 2]); grid on;
save_al_fig(h2, "Fig_Al_2.png");


% === FIG 3: Surface 3D Error ===
h3 = figure(3, 'visible', 'off');
surf(results2D.PHmesh, log10(results2D.AlTmesh), log10(results2D.massErrSurf));
xlabel('pH'); ylabel('log_{10}(AlT)'); zlabel('log_{10}(Error)');
title('Surface pH vs AlT (Error)');
view(45, 30); 
save_al_fig(h3, "Fig_Al_3.png");


% === FIG 4: Surface with Gray Zone ===
h4 = figure(4, 'visible', 'off');
threshold = results2D.threshold;
isTiny = results2D.isTiny;
Z_forLog = results2D.massErrSurf;
Z_forLog(isTiny) = threshold;
Zplot = log10(Z_forLog);
Xmesh = results2D.PHmesh;
Ymesh = log10(results2D.AlTmesh);

% Plot main surface
surf(Xmesh, Ymesh, Zplot);
shading interp; colormap('jet'); hcb=colorbar; title(hcb,'log(Err)');
hold on;

% Plot tiny errors in gray
Z2 = Zplot;
Z2(~isTiny) = NaN;
s2 = surf(Xmesh, Ymesh, Z2);
set(s2, 'FaceColor', [0.6 0.6 0.6], 'EdgeColor', 'none');

xlabel('pH'); ylabel('log_{10}(AlT)'); zlabel('log(Err)');
title('Convergence Map (<1e-20 in Gray)');
view(0, 90); 
axis tight;
save_al_fig(h4, "Fig_Al_4.png");

fprintf('Done! Images saved: Fig_Al_1 to Fig_Al_4\n');

Generating Aluminum figures...
Done! Images saved: Fig_Al_1 to Fig_Al_4


# Aluminum Speciation Results

| Summary (Fig 1) | Species Log (Fig 2) |
|:---:|:---:|
| ![F1](Fig_Al_1.png) | ![F2](Fig_Al_2.png) |

| 3D Error Surface (Fig 3) | Convergence Map (Fig 4) |
|:---:|:---:|
| ![F3](Fig_Al_3.png) | ![F4](Fig_Al_4.png) |